# Verify that $\mathrm{tasmin} \leq \mathrm{tasmax}$

In dataset NEX-GDDP-CMIP6 some values of tasmin are greater than the respective tasmax values in the same day and this is not allowed logically.

The problem was first identified by Jacopo in data bias adjusted by me, but with further checks it seems to originate from the original data and my bias adjustment simply accentuates the absolute errors.

Since both NEX and I apply variants of quantile mapping method for bias adjustment, the hypothesis is that NEX applied bias adjustment to tasmin and tasmax variables separately and did not check for coherence of these two values after the adjustment. This can be considered a bug in the dataset.

Note that this problem should not be dependent on spatial and temporal dimensions, since they are just slices of the same model outputs. In other words, if the problem is present even for a single point in space and time of the dataset, that means that the entire dataset is subject to it and only due to randomness other points may not present the problem.

In [1]:
import importlib
import math

import numpy
import xarray

import settings
import thesis

# Workaround to reload settings if this block is executed a second time but notebook is not restarted.
try:
    id(flag_reload)
    flag_reload = importlib.reload(settings)
except NameError:  # Raised if `flag_reload` is not defined, hence when block is executed the first time.
    flag_reload = settings

In [2]:
dir_data = settings.dir_data / settings.location

def print_wrong(tasmin, tasmax):
    dtr = tasmax - tasmin  # Should be always non negative.
    wrong = dtr.where((dtr < 0).compute(), other=numpy.nan)  # Set NaNs to use count.
    c = wrong.count().values  # Absolute number of non-physical occurences.
    print(c / math.prod(wrong.sizes.values()))  # Relative number of non-physical occurences.

## Check coherence in reference data

Reference data (i.e. ERA5) do not present the error because tasmin and tasmax are evaluated directly from hourly tas data.

In [3]:
chunks = {"latitude": 1, "longitude": 1, "time": -1}
ds_reference = xarray.open_dataset(dir_data / "preprocessed" / "reference.nc", chunks=chunks)

print("reference", f"({math.prod(ds_reference.sizes.values())} points)")
print_wrong(ds_reference["tasmin"], ds_reference["tasmax"])

reference (243090 points)
0.0


## Check coherence in all models after bias adjustment

Since the problem was initially identified in EC-Earth3 data after I applied quantile delta mapping algorithm, now I check which models present this problem among the ones I use in the thesis.

In [4]:
m_dim = "model"
ds_historical_ba = xarray.open_mfdataset(
    [dir_data / "adjusted" / f"historical_{m}.nc" for m in thesis.models],
    chunks=chunks,
    concat_dim=m_dim,
    combine="nested",
    combine_attrs="drop_conflicts"
)
ds_historical_ba = ds_historical_ba.chunk({m_dim: -1})
ds_historical_ba = ds_historical_ba.assign_coords({m_dim: list(thesis.models)})

s_dim = "scenario"
ds_scenario_ba = xarray.open_mfdataset(
    [[dir_data / "adjusted" / f"{s}_{m}.nc" for m in thesis.models] for s in thesis.scenario],
    chunks=chunks,
    concat_dim=[s_dim, m_dim],
    combine="nested",
    combine_attrs="drop_conflicts"
)
ds_scenario_ba = ds_scenario_ba.chunk({s_dim: 1, m_dim: -1})
ds_scenario_ba = ds_scenario_ba.assign_coords({s_dim: list(thesis.scenario), m_dim: list(thesis.models)})

In [5]:
print("historical", f"({math.prod(ds_historical_ba.isel(model=0).sizes.values())} points)")
for m in ds_historical_ba["model"].values:
    print(m)
    ds = ds_historical_ba.sel(model=m)
    print_wrong(ds["tasmin"], ds["tasmax"])
print()

for s in ds_scenario_ba["scenario"].values:
    print(s, f"({math.prod(ds_scenario_ba.sel(scenario=s).isel(model=0).sizes.values())} points)")
    for m in ds_scenario_ba["model"].values:
        print(m)
        ds = ds_scenario_ba.sel(scenario=s, model=m)
        print_wrong(ds["tasmin"], ds["tasmax"])
    print()

historical (496035 points)
ACCESS-CM2
4.031973550253511e-06
ACCESS-ESM1-5
2.4191841301521062e-05
BCC-CSM2-MR
2.0159867751267553e-06
CanESM5
0.0
CMCC-ESM2
0.0
CNRM-CM6-1
1.0079933875633775e-05
CNRM-ESM2-1
2.0159867751267553e-06
EC-Earth3
1.4111907425887287e-05
EC-Earth3-Veg-LR
8.063947100507021e-06
FGOALS-g3
0.0
GFDL-ESM4
0.0
INM-CM4-8
4.031973550253511e-06
INM-CM5-0
4.031973550253511e-06
IPSL-CM6A-LR
0.0
MIROC-ES2L
0.0
MIROC6
0.0
MPI-ESM1-2-HR
4.233572227766186e-05
MPI-ESM1-2-LR
0.0
MRI-ESM2-0
2.6207828076647818e-05
NorESM2-LM
0.0
NorESM2-MM
4.031973550253511e-06
TaiESM1
0.0

ssp126 (496035 points)
ACCESS-CM2
0.0
ACCESS-ESM1-5
6.047960325380266e-06
BCC-CSM2-MR
8.063947100507021e-06
CanESM5
0.0
CMCC-ESM2
0.0
CNRM-CM6-1
2.0159867751267553e-06
CNRM-ESM2-1
0.0
EC-Earth3
3.830374872740835e-05
EC-Earth3-Veg-LR
1.2095920650760531e-05
FGOALS-g3
0.0
GFDL-ESM4
0.0
INM-CM4-8
4.031973550253511e-06
INM-CM5-0
1.0079933875633775e-05
IPSL-CM6A-LR
0.0
MIROC-ES2L
0.0
MIROC6
0.0
MPI-ESM1-2-HR
3.225578840

## Check coherence for all models before bias adjustment

Is the cause of the problem the bias adjustment I apply to NEX-GDDP-CMIP6 data?
To answer this question, first I need to check that the original data did not present the problem.

Instead, as can be seen below, also NEX-GDDP-CMIP6 original data is affected by this problem (the preprocessing I applied introduces no changes in the values and their coordinates, for a proof check the content of file `debug_tas_extremes.txt` which is the same test as below but performed on data before they was preprocessed).

In [6]:
ds_historical = xarray.open_mfdataset(
    [dir_data / "preprocessed" / f"historical_{m}.nc" for m in thesis.models],
    chunks=chunks,
    concat_dim=m_dim,
    combine="nested",
    combine_attrs="drop_conflicts"
)
ds_historical = ds_historical.chunk({m_dim: -1})
ds_historical = ds_historical.assign_coords({m_dim: list(thesis.models)})

ds_scenario = xarray.open_mfdataset(
    [[dir_data / "preprocessed" / f"{s}_{m}.nc" for m in thesis.models] for s in thesis.scenario],
    chunks=chunks,
    concat_dim=[s_dim, m_dim],
    combine="nested",
    combine_attrs="drop_conflicts"
)
ds_scenario = ds_scenario.chunk({s_dim: 1, m_dim: -1})
ds_scenario = ds_scenario.assign_coords({s_dim: list(thesis.scenario), m_dim: list(thesis.models)})

In [7]:
print("historical", f"({math.prod(ds_historical.isel(model=0).sizes.values())} points)")
for m in ds_historical["model"].values:
    print(m)
    ds = ds_historical.sel(model=m)
    print_wrong(ds["tasmin"], ds["tasmax"])
print()

for s in ds_scenario["scenario"].values:
    print(s, f"({math.prod(ds_scenario.sel(scenario=s).isel(model=0).sizes.values())} points)")
    for m in ds_scenario["model"].values:
        print(m)
        ds = ds_scenario.sel(scenario=s, model=m)
        print_wrong(ds["tasmin"], ds["tasmax"])
    print()

historical (243090 points)
ACCESS-CM2
0.0
ACCESS-ESM1-5
0.0
BCC-CSM2-MR
0.0
CanESM5
0.0
CMCC-ESM2
0.0
CNRM-CM6-1
0.0
CNRM-ESM2-1
0.0
EC-Earth3
0.0
EC-Earth3-Veg-LR
0.0
FGOALS-g3
0.0
GFDL-ESM4
0.0
INM-CM4-8
4.11370274383973e-06
INM-CM5-0
4.11370274383973e-06
IPSL-CM6A-LR
0.0
MIROC-ES2L
0.0
MIROC6
0.0
MPI-ESM1-2-HR
0.0
MPI-ESM1-2-LR
0.0
MRI-ESM2-0
0.0
NorESM2-LM
4.11370274383973e-06
NorESM2-MM
0.0
TaiESM1
0.0

ssp126 (252945 points)
ACCESS-CM2
0.0
ACCESS-ESM1-5
0.0
BCC-CSM2-MR
0.0
CanESM5
0.0
CMCC-ESM2
0.0
CNRM-CM6-1
0.0
CNRM-ESM2-1
0.0
EC-Earth3
7.906857221925715e-06
EC-Earth3-Veg-LR
3.162742888770286e-05
FGOALS-g3
0.0
GFDL-ESM4
0.0
INM-CM4-8
0.0
INM-CM5-0
1.1860285832888572e-05
IPSL-CM6A-LR
0.0
MIROC-ES2L
0.0
MIROC6
0.0
MPI-ESM1-2-HR
1.1860285832888572e-05
MPI-ESM1-2-LR
0.0
MRI-ESM2-0
0.0
NorESM2-LM
0.0
NorESM2-MM
2.7674000276740002e-05
TaiESM1
0.0

ssp245 (252945 points)
ACCESS-CM2
0.0
ACCESS-ESM1-5
0.0
BCC-CSM2-MR
0.0
CanESM5
0.0
CMCC-ESM2
0.0
CNRM-CM6-1
0.0
CNRM-ESM2-1
0.0
EC-Earth3